# Information Gain-based Policy Optimization for Multi-Turn Search Agents

本 Notebook 复现 Wang et al., ICLR 2026（[arXiv:2510.14967](https://arxiv.org/abs/2510.14967)）中的 IGPO 算法核心流程。  
代码仓库：https://github.com/ankknaiii/igpo-agentic-search

当组内所有 rollout 的结果奖励相同时，纯结果驱动的 GRPO 会出现 advantage 坍缩。IGPO 通过相邻轮次 teacher-forced 真实答案概率差构造过程奖励，并与终局 F1 结果信号联合优化。

运行环境：T4 GPU。请按顺序执行以下单元格。

In [ ]:
#@title 环境配置
import os, sys, subprocess, zipfile
from pathlib import Path

REPO_URL = "https://github.com/ankknaiii/igpo-agentic-search.git"
ROOT = Path("/content/igpo-agentic-search")

def sh(cmd: str):
    print("$", cmd)
    subprocess.check_call(cmd, shell=True)

if not (ROOT / "igpo").exists():
    zip_path = Path("/content/igpo-agentic-search.zip")
    if zip_path.exists():
        with zipfile.ZipFile(zip_path) as z:
            z.extractall("/content")
        if not (ROOT / "igpo").exists() and Path("/content/igpo").exists():
            ROOT = Path("/content")
    else:
        if ROOT.exists():
            sh(f"rm -rf {ROOT}")
        sh(f"git clone --depth 1 {REPO_URL} {ROOT}")

assert (ROOT / "igpo").exists(), f"repository incomplete: {ROOT}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("cwd=", os.getcwd())
sh("pip install -q -r requirements.txt")
sh("pip install -q -e .")

In [ ]:
#@title 完整性校验
import subprocess
subprocess.check_call("python scripts/integrity_check.py", shell=True)
subprocess.check_call("pytest -q tests/", shell=True)

## 训练配置说明

| 字段 | 含义 |
|------|------|
| `model_name` | 基座模型；默认从 ModelScope 优先加载 |
| `algo` | `igpo` 或 `grpo` |
| `group_size` | 每问题 rollout 数 |
| `ppo_epochs` | 每 batch 的策略更新内循环次数 |
| `gamma` | turn-level advantage 折扣因子 |
| `eval_every` | held-out 评估间隔 |

In [ ]:
#@title 轻量级训练验证
import torch
from igpo.train.trainer import TrainConfig, run_training

print("device=", "cuda:" + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

cfg_check = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    model_source="auto",
    algo="igpo",
    max_steps=1,
    prompts_per_step=1,
    group_size=2,
    max_turns=2,
    max_new_tokens=64,
    ppo_epochs=2,
    learning_rate=1e-5,
    info_gain_type="prob_diff",
    info_gain_norm_mode="separate",
    output_dir="./outputs/check",
    eval_every=0,
)
hist_check = run_training(cfg_check)
hist_check[-1]

In [ ]:
#@title 训练执行
from igpo.train.trainer import TrainConfig, run_training

cfg = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    model_source="auto",
    algo="igpo",
    max_steps=8,
    prompts_per_step=2,
    group_size=4,
    max_turns=3,
    max_new_tokens=128,
    ppo_epochs=4,
    gamma=0.95,
    learning_rate=1e-5,
    info_gain_type="prob_diff",
    info_gain_norm_mode="separate",
    output_dir="./outputs/igpo_colab",
    eval_every=4,
    eval_samples=20,
)
history = run_training(cfg)
history[-1]

In [ ]:
#@title 评估与可视化
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

for fp in [
    "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
]:
    p = Path(fp)
    if p.exists():
        try:
            fm.fontManager.addfont(str(p))
        except Exception:
            pass

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "Noto Sans SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

steps = [h.step for h in history]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].plot(steps, [h.mean_f1 for h in history])
axes[0].set_title("平均 F1")
axes[1].plot(steps, [h.collapse_rate for h in history])
axes[1].set_title("结果奖励坍缩率")
axes[2].plot(steps, [h.mean_abs_ig for h in history])
axes[2].set_title("平均信息增益绝对值")
for ax in axes:
    ax.set_xlabel("训练步数")
plt.tight_layout()
plt.show()

eval_path = Path(cfg.output_dir) / "eval_metrics.jsonl"
if eval_path.exists():
    rows = [json.loads(x) for x in eval_path.read_text(encoding="utf-8").splitlines() if x.strip()]
    print("最近一次评估:", rows[-1] if rows else {})

In [ ]:
#@title 结果导出
from pathlib import Path
import shutil

export_dir = Path("./exports/igpo_run")
export_dir.mkdir(parents=True, exist_ok=True)
src = Path(cfg.output_dir)
for name in ["lora", "metrics.jsonl", "eval_metrics.jsonl", "config.json"]:
    p = src / name
    if p.exists():
        dst = export_dir / name
        if p.is_dir():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(p, dst)
        else:
            shutil.copy2(p, dst)
print("导出目录:", export_dir.resolve())
print("包含:", sorted(x.name for x in export_dir.iterdir()))

## 说明

- Advantage 坍缩：组内结果奖励完全相同时，z-score 后 advantage 为零。
- 过程奖励：$r_t = P(\mathrm{GT}\mid \mathrm{ctx}_t) - P(\mathrm{GT}\mid \mathrm{ctx}_{t-1})$，采用 teacher forcing。
- 信用分配：信息增益轮次与终局 F1 分别归一化后，按 $\gamma$ 折扣回传。
- 相对 MCTS / 外部奖励模型：监督信号内生于策略对真实答案的信念更新，标注成本更低。